In [4]:
# ================================================================
# AUTHENTICATED GITHUB AUDIT — cognitive-self-model
# هیچ Tokenی در Notebook، فایل یا Git remote ذخیره نمی‌شود.
# ================================================================

from pathlib import Path
import getpass
import os
import shutil
import stat
import subprocess
import textwrap

OWNER = "motiza345"
REPO  = "cognitive-self-model"
REPO_URL = f"https://github.com/{OWNER}/{REPO}.git"
TARGET = Path("/content/cognitive-self-model")

def run(cmd, cwd=None, check=False):
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(result.stdout.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result

print("=" * 90)
print("GITHUB AUTHENTICATED AUDIT")
print("=" * 90)

token = getpass.getpass(
    "GitHub Fine-grained PAT (input hidden; do NOT paste it in chat): "
).strip()

if not token:
    raise ValueError("Token was empty.")

# askpass موقت: Token نه در URL قرار می‌گیرد، نه در خروجی چاپ می‌شود.
askpass = Path("/tmp/github_askpass.py")
askpass.write_text(
    textwrap.dedent("""\
    #!/usr/bin/env python3
    import os
    import sys

    prompt = sys.argv[1].lower()
    if "username" in prompt:
        print("x-access-token")
    else:
        print(os.environ["GITHUB_PAT"])
    """),
    encoding="utf-8",
)
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)

env = os.environ.copy()
env["GITHUB_PAT"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    # اول وجود branchهای remote را با دسترسی معتبر تأیید می‌کنیم.
    print("\n[1/4] Remote branches:")
    result = subprocess.run(
        ["git", "ls-remote", "--heads", REPO_URL],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(result.stdout.rstrip())

    if result.returncode != 0:
        raise RuntimeError(
            "\nAccess failed. Check:\n"
            "1) Token belongs to GitHub account motiza345\n"
            "2) Token has access to this exact repository\n"
            "3) Contents permission is Read (or Read and write)\n"
            "4) Repository name/owner are correct\n"
        )

    # جلوگیری از مخلوط‌شدن با فایل‌های موقتی یا clone ناقص قبلی
    if TARGET.exists():
        backup = Path("/content/cognitive-self-model_pre_audit_backup")
        if backup.exists():
            shutil.rmtree(backup)
        shutil.move(str(TARGET), str(backup))
        print(f"\nExisting local directory moved safely to:\n{backup}")

    print("\n[2/4] Cloning repository...")
    clone = subprocess.run(
        ["git", "clone", REPO_URL, str(TARGET)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(clone.stdout.rstrip())
    if clone.returncode != 0:
        raise RuntimeError("Clone failed.")

    # اطمینان: origin بدون credential باقی می‌ماند.
    subprocess.run(
        ["git", "remote", "set-url", "origin", REPO_URL],
        cwd=str(TARGET),
        env=env,
        check=True,
    )

    print("\n[3/4] Git state:")
    run(["git", "remote", "-v"], cwd=TARGET)
    run(["git", "branch", "-a"], cwd=TARGET)
    run(["git", "status", "--short", "--branch"], cwd=TARGET)

    print("\n[4/4] Complete commit history:")
    run(
        ["git", "log", "--all", "--graph", "--decorate", "--oneline", "-n", "100"],
        cwd=TARGET,
    )

    print("\nRepository files:")
    run(
        ["bash", "-lc", "find . -type f -not -path './.git/*' | sort | sed 's#^./##'"],
        cwd=TARGET,
    )

    print("\n" + "=" * 90)
    print("AUDIT COMPLETED")
    print(f"Working directory: {TARGET}")
    print("Remote URL was sanitized; no token was stored in Git config.")
    print("=" * 90)

finally:
    # پاک‌سازی فوری Token و helper موقت از محیط Runtime
    token = None
    env.pop("GITHUB_PAT", None)
    if askpass.exists():
        askpass.unlink()

GITHUB AUTHENTICATED AUDIT
GitHub Fine-grained PAT (input hidden; do NOT paste it in chat): ··········

[1/4] Remote branches:
8ace58a93ef534177f520de0da574677be4395ea	refs/heads/main

[2/4] Cloning repository...
Cloning into '/content/cognitive-self-model'...

[3/4] Git state:
origin	https://github.com/motiza345/cognitive-self-model.git (fetch)
origin	https://github.com/motiza345/cognitive-self-model.git (push)
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
## main...origin/main

[4/4] Complete commit history:
* 8ace58a (HEAD -> main, origin/main, origin/HEAD) Initialize cognitive self-model repository structure

Repository files:
artifacts/.gitkeep
configs/.gitkeep
docs/.gitkeep
.gitignore
legacy/.gitkeep
notebooks/.gitkeep
README.md
scripts/.gitkeep
src/cognitive_self_model/__init__.py
tests/.gitkeep

AUDIT COMPLETED
Working directory: /content/cognitive-self-model
Remote URL was sanitized; no token was stored in Git config.
